# ЛР-02: Ротация топливного резерва

## Worked example: military 02

Это полностью разобранный example. Он показывает образец постановки, решения и интерпретации транспортной модели.

## 1. Постановка кейса

Открытая задача: часть топлива уходит в резерв через фиктивного потребителя.

### Запасы

| Поставщик | Объём |
| --- | --- |
| Хранилище A | 42 |
| Хранилище B | 28 |
| Хранилище C | 20 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Часть 1 | 20 |
| Часть 2 | 18 |
| Часть 3 | 16 |
| Часть 4 | 14 |

### Матрица затрат

| Откуда / Куда | Часть 1 | Часть 2 | Часть 3 | Часть 4 |
| --- | --- | --- | --- | --- |
| Хранилище A | 4 | 7 | 8 | 9 |
| Хранилище B | 5 | 4 | 6 | 7 |
| Хранилище C | 7 | 6 | 4 | 5 |

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

def balance_transport_problem(supplies, demands, costs, supplier_names, consumer_names):
    supplies = supplies.astype(float).copy()
    demands = demands.astype(float).copy()
    costs = costs.astype(float).copy()
    supplier_names = list(supplier_names)
    consumer_names = list(consumer_names)

    diff = supplies.sum() - demands.sum()
    if diff > 0:
        demands = np.append(demands, diff)
        consumer_names.append('Фиктивный потребитель')
        costs = np.column_stack([costs, np.zeros(len(supplies))])
    elif diff < 0:
        supplies = np.append(supplies, -diff)
        supplier_names.append('Фиктивный поставщик')
        costs = np.vstack([costs, np.zeros(len(demands))])

    return supplies, demands, costs, supplier_names, consumer_names

def solve_transport_problem(supplies, demands, costs):
    m, n = costs.shape
    c = costs.flatten()

    A_eq = []
    b_eq = []

    for i in range(m):
        row = np.zeros(m * n)
        row[i * n:(i + 1) * n] = 1
        A_eq.append(row)
        b_eq.append(supplies[i])

    for j in range(n):
        row = np.zeros(m * n)
        row[j::n] = 1
        A_eq.append(row)
        b_eq.append(demands[j])

    result = linprog(
        c,
        A_eq=np.array(A_eq),
        b_eq=np.array(b_eq),
        bounds=[(0, None)] * (m * n),
        method='highs',
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result, result.x.reshape(m, n)


In [2]:
supplier_names = ['Хранилище A', 'Хранилище B', 'Хранилище C']
consumer_names = ['Часть 1', 'Часть 2', 'Часть 3', 'Часть 4']
supplies = np.array([42, 28, 20], dtype=float)
demands = np.array([20, 18, 16, 14], dtype=float)
costs = np.array([[4, 7, 8, 9], [5, 4, 6, 7], [7, 6, 4, 5]], dtype=float)

balanced_supplies, balanced_demands, balanced_costs, supplier_names, consumer_names = balance_transport_problem(
    supplies, demands, costs, supplier_names, consumer_names
)

result, plan = solve_transport_problem(balanced_supplies, balanced_demands, balanced_costs)

plan_df = pd.DataFrame(plan, index=supplier_names, columns=consumer_names)
cost_df = pd.DataFrame(balanced_costs, index=supplier_names, columns=consumer_names)

print('Оптимальная стоимость:', round(result.fun, 2))
print()
print('План перевозок:')
display(plan_df)
print('Матрица затрат:')
display(cost_df)


Оптимальная стоимость: 306.0

План перевозок:


,Часть 1,Часть 2,Часть 3,Часть 4,Фиктивный потребитель
Хранилище A,20.0,0.0,0.0,0.0,22.0
Хранилище B,0.0,18.0,0.0,10.0,-0.0
Хранилище C,0.0,0.0,16.0,4.0,0.0


Матрица затрат:


,Часть 1,Часть 2,Часть 3,Часть 4,Фиктивный потребитель
Хранилище A,4.0,7.0,8.0,9.0,0.0
Хранилище B,5.0,4.0,6.0,7.0,0.0
Хранилище C,7.0,6.0,4.0,5.0,0.0


In [3]:
used_routes = []
for supplier in plan_df.index:
    for consumer in plan_df.columns:
        value = float(plan_df.loc[supplier, consumer])
        if value > 1e-9:
            used_routes.append({
                'маршрут': f'{supplier} -> {consumer}',
                'объём': round(value, 2),
                'тариф': float(cost_df.loc[supplier, consumer]),
                'затраты': round(value * float(cost_df.loc[supplier, consumer]), 2),
            })

used_routes_df = pd.DataFrame(used_routes)
print('Использованные маршруты:')
display(used_routes_df)
print('Проверка баланса по поставщикам:')
display(pd.DataFrame({'план': plan_df.sum(axis=1), 'запас': balanced_supplies}, index=plan_df.index))
print('Проверка баланса по потребителям:')
display(pd.DataFrame({'план': plan_df.sum(axis=0), 'спрос': balanced_demands}, index=plan_df.columns))


Использованные маршруты:


,маршрут,объём,тариф,затраты
0,Хранилище A -> Часть 1,20.0,4.0,80.0
1,Хранилище A -> Фиктивный потребитель,22.0,0.0,0.0
2,Хранилище B -> Часть 2,18.0,4.0,72.0
3,Хранилище B -> Часть 4,10.0,7.0,70.0
4,Хранилище C -> Часть 3,16.0,4.0,64.0
5,Хранилище C -> Часть 4,4.0,5.0,20.0


Проверка баланса по поставщикам:


,план,запас
Хранилище A,42.0,42.0
Хранилище B,28.0,28.0
Хранилище C,20.0,20.0


Проверка баланса по потребителям:


,план,спрос
Часть 1,20.0,20.0
Часть 2,18.0,18.0
Часть 3,16.0,16.0
Часть 4,14.0,14.0
Фиктивный потребитель,22.0,22.0


## 2. Что важно проговорить в выводе

- фиктивный потребитель отражает не потери, а плановый резерв;
- маршруты с меньшей стоимостью забирают на себя основную нагрузку;
- баланс нужно проверять уже после расширения модели.

Для отчёта обычно достаточно перечислить ненулевые маршруты, пояснить роль фиктивного узла, если он появился, и отдельно указать итоговую стоимость перевозок.